# data_quality_suite.ipynb v1.3

**Run order:** standalone — run anytime after the silver/gold materialisation notebooks. Not a dependency of any other notebook.

Runs the same two-tier data quality checks as `tests/data_quality/run_checks.py` — Tier 2 hand-written checks (`tests/data_quality/checks/*.sql`) and Tier 1 registry-driven checks (`tests/data_quality/registry/tier1_*.yaml`, one file per layer) — against `workspace.genealogy`, and writes results to `genealogy.data_quality_results` — the same Delta table the Python runner writes to, so trend data from both surfaces lives in one place.

**No Asana integration here.** CI (`run_checks.py` via GitHub Actions) is the sole owner of Asana task filing, so this notebook can be run freely — on a schedule or ad hoc — without racing CI to create duplicate tasks. This path is for native/scheduled runs and trend data, not alerting.

**Latest changes (v1.3):** Test Plan Phase 3 added bronze-layer (`tier1_bronze_registry.yaml`) and ref-layer (`tier1_ref_registry.yaml`) registries. Bronze reuses Phase 1's `row_count_not_zero`/`freshness_vs_source` check types; ref adds one new check type, `not_blank` (NULL or empty-string), alongside Phase 2's `not_null`/`uniqueness`/`fk_integrity`. No changes needed to Cell 1's glob (it already picks up any `tier1_*.yaml` file) — only Cell 3's `build_tier1_check_sql`/`tier1_check_label`.

## Cell 1 — Locate the checks directory and registry files
## Cell 2 — Tier 2: parse header + run each `.sql` check
## Cell 3 — Tier 1: run registry-driven checks (all layers)
## Cell 4 — Sync the registry table, write results to `data_quality_results`
## Cell 5 — Verification: pass/fail summary

## Cell 1 — Locate the checks directory and registry files

This notebook lives at the repo root; the Tier 2 check files live at `tests/data_quality/checks/` and the Tier 1 registry seeds at `tests/data_quality/registry/tier1_*.yaml` (one per layer — gold, silver, more to follow in later phases), both in the same Git-Repos checkout. Databricks Repos expose the checkout at `/Workspace` + the notebook's workspace path for direct file access (DBR 11.2+). If that mapping is wrong for this workspace, set the `checks_dir_override` widget instead of guessing further — the cell below fails loudly with the path it tried, rather than silently finding zero checks.


In [ ]:
dbutils.widgets.text("checks_dir_override", "", "Checks dir override (optional)")
override = dbutils.widgets.get("checks_dir_override").strip()

import glob
import os

if override:
    CHECKS_DIR = override
    REGISTRY_DIR = os.path.join(os.path.dirname(override.rstrip("/")), "registry")
else:
    notebook_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    repo_root = "/Workspace" + os.path.dirname(notebook_path)
    CHECKS_DIR = f"{repo_root}/tests/data_quality/checks"
    REGISTRY_DIR = f"{repo_root}/tests/data_quality/registry"

if not os.path.isdir(CHECKS_DIR):
    raise FileNotFoundError(
        f"Checks directory not found at '{CHECKS_DIR}'. "
        "Set the checks_dir_override widget to the correct path for this workspace."
    )

registry_paths = sorted(glob.glob(os.path.join(REGISTRY_DIR, "tier1_*.yaml")))
if not registry_paths:
    raise FileNotFoundError(
        f"No tier1_*.yaml registry files found under '{REGISTRY_DIR}'. "
        "Check the repo checkout layout if checks_dir_override was needed above."
    )

check_files = sorted(
    os.path.join(CHECKS_DIR, f) for f in os.listdir(CHECKS_DIR) if f.endswith(".sql")
)
print(f"CHECKS_DIR = {CHECKS_DIR}")
print(f"Found {len(check_files)} Tier 2 check file(s):")
for f in check_files:
    print(" ", os.path.basename(f))
print(f"Found {len(registry_paths)} Tier 1 registry file(s):")
for f in registry_paths:
    print(" ", os.path.basename(f))


## Cell 2 — Tier 2: parse header + run each `.sql` check

Same tolerant line-prefix header parser as `run_checks.py` (kept in sync manually — there are only two small parsers, one per runtime, both reading the same `.sql` files, so the check *logic* itself is never duplicated). Each check's query is run via `spark.sql` rather than the `databricks-sql-connector` path the Python runner uses, since this notebook already has a live Spark session.

Results accumulate into the shared `results` / `result_rows` lists — Cell 3 appends Tier 1 results to the same lists, and Cell 4 writes both tiers to `data_quality_results` together in one run.


In [ ]:
import json
import re
import uuid
from datetime import datetime, timezone
from decimal import Decimal

HEADER_KEYS = {
    "id", "title", "severity", "guards_bug", "known_failing",
    "existing_asana_task", "description",
}
SAMPLE_CAP = 20


def parse_check_file(path):
    with open(path) as fh:
        lines = fh.read().splitlines()
    header = {k: "" for k in HEADER_KEYS}
    current_key = None
    sql_start = len(lines)

    for i, line in enumerate(lines):
        if not line.strip():
            sql_start = i + 1
            break
        if not line.startswith("--"):
            sql_start = i
            break
        content = line[2:]
        if content.startswith(" "):
            content = content[1:]
        match = re.match(r"^(\w+):\s?(.*)$", content)
        if match and match.group(1) in HEADER_KEYS:
            current_key = match.group(1)
            value = match.group(2).strip()
            header[current_key] = "" if (current_key == "description" and value == ">") else value
        elif current_key == "description":
            header["description"] = (header["description"] + " " + content.strip()).strip()

    sql_text = "\n".join(lines[sql_start:]).strip()
    return {
        "id": header["id"].strip(),
        "title": header["title"].strip(),
        "severity": header["severity"].strip().lower(),
        "guards_bug": header["guards_bug"].strip() or None,
        "known_failing": header["known_failing"].strip().lower() == "true",
        "existing_asana_task": header["existing_asana_task"].strip() or None,
        "description": header["description"].strip(),
        "sql": sql_text,
    }


def json_default(value):
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, datetime):
        return value.isoformat()
    return str(value)


run_id = str(uuid.uuid4())
run_at = datetime.now(timezone.utc)
results = []
result_rows = []

for path in check_files:
    check = parse_check_file(path)
    df = spark.sql(check["sql"])
    rows = [r.asDict(recursive=True) for r in df.collect()]
    violation_count = len(rows)
    status = "pass" if violation_count == 0 else "fail"
    sample_violations = (
        json.dumps(rows[:SAMPLE_CAP], default=json_default) if status == "fail" else None
    )

    results.append({**check, "status": status, "violation_count": violation_count})
    result_rows.append(dict(
        run_id=run_id,
        check_id=check["id"],
        run_at=run_at,
        status=status,
        severity=check["severity"],
        violation_count=violation_count,
        sample_violations=sample_violations,
        known_failing=check["known_failing"],
        guards_bug=check["guards_bug"],
        asana_task_gid=None,  # this notebook never files Asana tasks -- see header note
    ))
    print(f"{check['id']:<8} {status:<6} {violation_count:>6} rows  ({check['severity']})")


## Cell 3 — Tier 1: run registry-driven checks (all layers)

Mirrors `build_tier1_check_sql`/`tier1_check_label` in `tests/data_quality/run_checks.py` (kept
in sync manually, same duplication tradeoff already accepted for the Tier 2
parser above — one Python runtime per surface, same generated SQL). Reads
every `registry/tier1_*.yaml` file directly rather than
`genealogy.ref_data_quality_registry` — git is the source of truth, the
Delta table is a queryable materialization of it (synced in Cell 4, after
these checks run, not before).

Six check types across the four registries so far:
- `row_count_not_zero` (gold, bronze) — the object has at least one row.
- `freshness_vs_source` (gold, bronze) — the table's last Delta write isn't older
  than any of its `depends_on` tables. Never applies to a `view` object and
  never names a `view` in `depends_on` — `DESCRIBE HISTORY` has no Delta
  log to read for a view.
- `not_null` (silver, ref) — a single column is never NULL.
- `not_blank` (ref) — a single column is never NULL or an empty/whitespace
  string (`TRIM(column) = ''`) — distinct from `not_null` because ref-layer
  keys are hand-typed strings, where a blank string is a real, separate
  risk from a NULL.
- `uniqueness` (silver, ref) — a column or column-list (composite key) has
  no duplicate values.
- `fk_integrity` (silver) — every non-NULL value in a column resolves to a
  row in another table's reference column.


In [ ]:
import yaml

registry_objects = []
for registry_path in registry_paths:
    with open(registry_path) as fh:
        registry_objects.extend(yaml.safe_load(fh)["objects"])


def build_tier1_check_sql(check_type, object_name, check_cfg):
    if check_type == "row_count_not_zero":
        return f"SELECT 'EMPTY_TABLE' AS violation FROM (SELECT COUNT(*) AS n FROM {object_name}) t WHERE t.n = 0"

    if check_type == "freshness_vs_source":
        source_union = "\n    UNION ALL\n    ".join(
            f"SELECT timestamp AS ts FROM (DESCRIBE HISTORY {src})" for src in check_cfg.get("depends_on", [])
        )
        return (
            "WITH target AS (\n"
            f"  SELECT MAX(timestamp) AS last_write FROM (DESCRIBE HISTORY {object_name})\n"
            "),\n"
            "source AS (\n"
            "  SELECT MAX(ts) AS last_write FROM (\n"
            f"    {source_union}\n"
            "  )\n"
            ")\n"
            "SELECT target.last_write AS target_last_write, source.last_write AS source_last_write\n"
            "FROM target, source\n"
            "WHERE target.last_write < source.last_write"
        )

    if check_type == "not_null":
        return f"SELECT * FROM {object_name} WHERE {check_cfg['column']} IS NULL"

    if check_type == "not_blank":
        column = check_cfg["column"]
        return f"SELECT * FROM {object_name} WHERE {column} IS NULL OR TRIM({column}) = ''"

    if check_type == "uniqueness":
        columns = ", ".join(check_cfg["columns"])
        return (
            f"SELECT {columns}, COUNT(*) AS dupe_count FROM {object_name} "
            f"GROUP BY {columns} HAVING COUNT(*) > 1"
        )

    if check_type == "fk_integrity":
        column = check_cfg["column"]
        return (
            f"SELECT DISTINCT {column} FROM {object_name} t "
            f"WHERE t.{column} IS NOT NULL AND NOT EXISTS "
            f"(SELECT 1 FROM {check_cfg['ref_table']} r WHERE r.{check_cfg['ref_column']} = t.{column})"
        )

    raise ValueError(f"unknown Tier 1 check_type: {check_type}")


def tier1_check_label(check_type, check_cfg):
    if check_type == "not_null":
        return check_cfg["column"]
    if check_type == "not_blank":
        return check_cfg["column"]
    if check_type == "uniqueness":
        return "_".join(check_cfg["columns"])
    if check_type == "fk_integrity":
        return check_cfg["column"]
    return None


tier1_check_count = 0
for obj in registry_objects:
    short_name = obj["name"].split(".")[-1].upper()
    for check_cfg in obj["checks"]:
        check_type = check_cfg["check_type"]
        label = tier1_check_label(check_type, check_cfg)
        id_suffix = f"-{label.upper()}" if label else ""
        title_suffix = f" ({label})" if label else ""
        check = {
            "id": f"T1-{check_type.upper()}-{short_name}{id_suffix}",
            "title": f"{obj['name']} {check_type.replace('_', ' ')}{title_suffix}",
            "severity": check_cfg["severity"].strip().lower(),
            "guards_bug": None,
            "known_failing": check_cfg.get("known_failing", False),
            "existing_asana_task": check_cfg.get("existing_asana_task"),
        }
        sql_text = build_tier1_check_sql(check_type, obj["name"], check_cfg)

        df = spark.sql(sql_text)
        rows = [r.asDict(recursive=True) for r in df.collect()]
        violation_count = len(rows)
        status = "pass" if violation_count == 0 else "fail"
        sample_violations = (
            json.dumps(rows[:SAMPLE_CAP], default=json_default) if status == "fail" else None
        )

        results.append({**check, "status": status, "violation_count": violation_count})
        result_rows.append(dict(
            run_id=run_id,
            check_id=check["id"],
            run_at=run_at,
            status=status,
            severity=check["severity"],
            violation_count=violation_count,
            sample_violations=sample_violations,
            known_failing=check["known_failing"],
            guards_bug=check["guards_bug"],
            asana_task_gid=None,
        ))
        tier1_check_count += 1
        print(f"{check['id']:<55} {status:<6} {violation_count:>6} rows  ({check['severity']})")

print(f"\n{tier1_check_count} Tier 1 registry check(s) run.")


## Cell 4 — Sync the registry table, write results to `data_quality_results`

Mirrors `ensure_registry_table`/`sync_registry_table` in `run_checks.py`,
including the same schema-migration step: Phase 1 created
`ref_data_quality_registry` without the `column_name`/`columns`/`ref_table`/
`ref_column` fields Phase 2's silver check types need, so this cell backfills
them with `ALTER TABLE ... ADD COLUMNS` if they're missing before syncing —
harmless (a no-op ALTER on `column_name`/etc. once they exist) on a
notebook run that follows a `run_checks.py` run that already added them, or
vice versa.

`genealogy.ref_data_quality_registry` is fully overwritten from the
checked-in YAML on every run (`overwrite` mode, not append) — it's a
queryable mirror of the registry files, not independent state.
`data_quality_results` stays `append` — every run adds a new `run_id`'s
worth of rows, same as the Python runner and as this notebook did before.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType, LongType, TimestampType
)

spark.sql("""
  CREATE TABLE IF NOT EXISTS genealogy.ref_data_quality_registry (
    object_name STRING,
    object_type STRING,
    check_type STRING,
    column_name STRING,
    columns STRING,
    depends_on STRING,
    ref_table STRING,
    ref_column STRING,
    severity STRING,
    known_failing BOOLEAN,
    existing_asana_task STRING,
    notes STRING
  ) USING DELTA
""")

existing_registry_columns = {
    row["col_name"] for row in spark.sql("DESCRIBE TABLE genealogy.ref_data_quality_registry").collect()
}
missing_registry_columns = {
    "column_name": "STRING", "columns": "STRING", "ref_table": "STRING", "ref_column": "STRING",
} 
missing_registry_columns = {
    name: type_ for name, type_ in missing_registry_columns.items() if name not in existing_registry_columns
}
if missing_registry_columns:
    cols_sql = ", ".join(f"{name} {type_}" for name, type_ in missing_registry_columns.items())
    spark.sql(f"ALTER TABLE genealogy.ref_data_quality_registry ADD COLUMNS ({cols_sql})")

registry_rows = [
    Row(
        object_name=obj["name"],
        object_type=obj["type"],
        check_type=check_cfg["check_type"],
        column_name=check_cfg.get("column"),
        columns=",".join(check_cfg.get("columns", [])) or None,
        depends_on=",".join(check_cfg.get("depends_on", [])) or None,
        ref_table=check_cfg.get("ref_table"),
        ref_column=check_cfg.get("ref_column"),
        severity=check_cfg["severity"].strip().lower(),
        known_failing=check_cfg.get("known_failing", False),
        existing_asana_task=check_cfg.get("existing_asana_task"),
        notes=check_cfg.get("notes"),
    )
    for obj in registry_objects
    for check_cfg in obj["checks"]
]
registry_schema = StructType([
    StructField("object_name", StringType()),
    StructField("object_type", StringType()),
    StructField("check_type", StringType()),
    StructField("column_name", StringType()),
    StructField("columns", StringType()),
    StructField("depends_on", StringType()),
    StructField("ref_table", StringType()),
    StructField("ref_column", StringType()),
    StructField("severity", StringType()),
    StructField("known_failing", BooleanType()),
    StructField("existing_asana_task", StringType()),
    StructField("notes", StringType()),
])
spark.createDataFrame(registry_rows, schema=registry_schema).write.mode("overwrite").saveAsTable(
    "genealogy.ref_data_quality_registry"
)
print(f"Synced {len(registry_rows)} row(s) to genealogy.ref_data_quality_registry from {len(registry_paths)} registry file(s)")

spark.sql("""
  CREATE TABLE IF NOT EXISTS genealogy.data_quality_results (
    run_id STRING,
    check_id STRING,
    run_at TIMESTAMP,
    status STRING,
    severity STRING,
    violation_count BIGINT,
    sample_violations STRING,
    known_failing BOOLEAN,
    guards_bug STRING,
    asana_task_gid STRING
  ) USING DELTA
""")

results_schema = StructType([
    StructField("run_id", StringType()),
    StructField("check_id", StringType()),
    StructField("run_at", TimestampType()),
    StructField("status", StringType()),
    StructField("severity", StringType()),
    StructField("violation_count", LongType()),
    StructField("sample_violations", StringType()),
    StructField("known_failing", BooleanType()),
    StructField("guards_bug", StringType()),
    StructField("asana_task_gid", StringType()),
])
spark.createDataFrame([Row(**r) for r in result_rows], schema=results_schema).write.mode("append").saveAsTable(
    "genealogy.data_quality_results"
)
print(f"run_id: {run_id} — wrote {len(result_rows)} row(s) to genealogy.data_quality_results "
      f"({len(check_files)} Tier 2 + {tier1_check_count} Tier 1)")


## Cell 5 — Verification


In [0]:
# Pass/fail summary for this run
passed = [r for r in results if r["status"] == "pass"]
failed = [r for r in results if r["status"] == "fail"]
newly_failed_critical = [
    r for r in failed if r["severity"] == "critical" and not r["known_failing"]
]

print(f"Checks run:            {len(results)}")
print(f"Passed:                {len(passed)}")
print(f"Failed:                {len(failed)}")
print(f"  of which known_failing (expected open bugs): {sum(1 for r in failed if r['known_failing'])}")
print(f"  of which NEW critical failures (expect 0):   {len(newly_failed_critical)}")

if newly_failed_critical:
    print("\nNEW critical failures — investigate and file/update an Asana task via the CI runner:")
    for r in newly_failed_critical:
        print(f"  {r['id']}: {r['title']} ({r['violation_count']} violating rows)")

spark.sql(f"""
  SELECT check_id, status, severity, violation_count, known_failing
  FROM genealogy.data_quality_results
  WHERE run_id = '{run_id}'
  ORDER BY check_id
""").display()
